# Aleksei Sorokin PhD Thesis 2025: Quasi-Monte Carlo Codes

Original QMCPy demo: [`QMCPy/demos/talk_paper_demos/SorokinThesis2025/sorokin_thesis_2025.ipynb`](../../../../QMCPy/demos/talk_paper_demos/SorokinThesis2025/sorokin_thesis_2025.ipynb)

This Julia translation keeps the same two themes as the Python notebook: point-set generation and kernel-method building blocks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/talk_paper_demos/SorokinThesis2025/sorokin_thesis_2025.ipynb)

Parity note: this archival Julia companion preserves the cited source and the figure-defining themes of point-set generation and kernel-method building blocks, while documenting the thesis-oriented presentation divergence.


In [1]:
using QMC
using Statistics
using Printf


## Point Sets

Generate lattice, digital-net, and Halton point sets in a moderately high-dimensional setting and check a few basic summaries.


In [2]:
d = 52
n = 2^8

lattice = Lattice(d; seed=7)
digital_net = DigitalNetB2(d; seed=7)
halton = Halton(d; seed=7)

X_lattice = gen_samples(lattice, n)
X_digital = gen_samples(digital_net, n)
X_halton = gen_samples(halton, n)

point_set_rows = [
    (name="Lattice",      size=size(X_lattice), sample_mean=mean(X_lattice)),
    (name="DigitalNetB2", size=size(X_digital), sample_mean=mean(X_digital)),
    (name="Halton",       size=size(X_halton),  sample_mean=mean(X_halton)),
]

@printf("  %-14s  %-12s  %s\n", "Name", "Size", "sample_mean")
println("  " * repeat("─", 40))
for row in point_set_rows
    @printf("  %-14s  (%d, %d)     %.4f\n", row.name, row.size[1], row.size[2], row.sample_mean)
end

@assert all(row.size == (n, d) for row in point_set_rows)
@assert all(abs(row.sample_mean - 0.5) < 0.02 for row in point_set_rows)

  Name            Size          sample_mean
  ────────────────────────────────────────


  Lattice         (256, 52)     0.5001


  DigitalNetB2    (256, 52)     0.5001
  Halton          (256, 52)     0.4998


## Kernel Methods

### Lattice + FFTBR + IFFTBR

The thesis notebook demonstrates fast transforms as the computational backbone for kernel methods on structured point sets. In `QMC.jl`, `bro_fft` and `bro_ifft` give the lattice-side round trip.


In [3]:
signal = collect(1.0:16.0)
coeffs = bro_fft(signal)
recovered_signal = bro_ifft(coeffs)
@printf("FFTBR/IFFTBR round-trip max error = %.3e
", maximum(abs.(signal - recovered_signal)))

λ_lattice = compute_kernel_eigenvalues(KernelShiftInvar(order=2), gen_samples(Lattice(3; seed=7), 64))
@printf("Lattice kernel eigenvalue summary: min = %.3e, max = %.3f
", minimum(λ_lattice), maximum(λ_lattice))

@assert maximum(abs.(signal - recovered_signal)) < 1e-12


FFTBR/IFFTBR round-trip max error = 0.000e+00
Lattice kernel eigenvalue summary: min = -2.186e-01, max = 64.000


### Digital Net + FWHT

For digital nets, the fast Walsh-Hadamard transform plays the analogous role. Applying `fwht` twice and dividing by the vector length recovers the original signal.


In [4]:
walsh_signal = collect(1.0:16.0)
walsh_coeffs = fwht(walsh_signal)
recovered_walsh = fwht(walsh_coeffs) ./ length(walsh_signal)
μ_digital = compute_kernel_eigenvalues(KernelDigShiftInvar(order=2), gen_samples(DigitalNetB2(3; seed=7), 64))
@printf("FWHT round-trip max error = %.3e
", maximum(abs.(walsh_signal - recovered_walsh)))
@printf("Digital-net kernel eigenvalue summary: min = %.3e, max = %.3f
", minimum(μ_digital), maximum(μ_digital))

@assert maximum(abs.(walsh_signal - recovered_walsh)) < 1e-12
@assert all(μ_digital .>= -1e-12)


FWHT round-trip max error = 0.000e+00
Digital-net kernel eigenvalue summary: min = 1.501e-11, max = 57.871


## Integration

The QMCPy thesis notebook continues with broader integration figures after the point-set and kernel-method material. The checked-in Julia archival notebook keeps the figure-defining point-set and transform sections executable here and tracks the later thesis examples through the dedicated Julia demos listed below.

## Cantilever Beam

The cantilever-beam sensitivity material is represented in the maintained Julia demo set through the vectorized and sensitivity notebooks rather than being duplicated in this archival thesis notebook.

## Bayesian Logistic Regression

The Bayesian-logistic-regression section from the QMCPy thesis notebook remains an archival parity target; the checked-in Julia notebook stays focused on the point-set and fast-transform figures that are already stable in `QMC.jl`.

## Ishigami Sensitivity Indices

Ishigami sensitivity examples are available in the Julia demo set, but they are not re-executed in this compact archival notebook.

## Neural Network Classifier Sensitiviy Indices

Likewise, the neural-network sensitivity material remains thesis-context documentation rather than a checked-in executable Julia section here.
